# Лабораторная работа №5: Рекомендательные системы


In [1]:
import importlib.util
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from surprise import Dataset

spec = importlib.util.spec_from_file_location("slim_impl", "src/SLIM.py")
slim_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(slim_mod)
MySLIM = slim_mod.SLIM

from SLIM import SLIM as SLIMRef, SLIMatrix

print("Все импорты успешны")

Все импорты успешны


## 1. Загрузка и исследование датасета

In [2]:
data = Dataset.load_builtin("ml-100k")
df = pd.DataFrame(data.raw_ratings, columns=["user_id", "item_id", "rating", "timestamp"])
df["rating"] = df["rating"].astype(float)

nu = df["user_id"].nunique()
ni = df["item_id"].nunique()
print(f"Пользователей: {nu}")
print(f"Фильмов:       {ni}")
print(f"Оценок:        {len(df)}")
print(f"Разреженность: {1 - len(df)/(nu*ni):.2%}")
print(f"Средняя оценка: {df.rating.mean():.2f}")
df.head()

Пользователей: 943
Фильмов:       1682
Оценок:        100000
Разреженность: 93.70%
Средняя оценка: 3.53


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


## 2. Предобработка

In [3]:
user_ids = sorted(df["user_id"].unique())
item_ids = sorted(df["item_id"].unique())
user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {v: i for i, v in enumerate(item_ids)}

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

n_users = len(user_ids)
n_items = len(item_ids)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

R_train = np.zeros((n_users, n_items))
for _, row in train_df.iterrows():
    R_train[int(row["user_idx"]), int(row["item_idx"])] = row["rating"]

R_test = np.zeros((n_users, n_items))
for _, row in test_df.iterrows():
    R_test[int(row["user_idx"]), int(row["item_idx"])] = row["rating"]

test_users   = test_df["user_idx"].values.astype(int)
test_items   = test_df["item_idx"].values.astype(int)
test_ratings = test_df["rating"].values

print(f"Матрица оценок: {R_train.shape}")

Train: 80000, Test: 20000
Матрица оценок: (943, 1682)


## 3. Своя реализация SLIM



In [4]:
slim = MySLIM(alpha=0.00212, l1_ratio=0.5, max_iter=50, tol=1e-7)


t0 = time.time()
slim.fit(R_train)
our_time = time.time() - t0

print(f"Время обучения: {our_time:.1f} сек ({our_time/60:.1f} мин)")

SLIM training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1682/1682 [00:00<00:00, 2009.19it/s]

Время обучения: 0.9 сек (0.0 мин)


In [5]:
R_pred_ours = slim.predict_all(R_train)

ours_preds = np.clip(R_pred_ours[test_users, test_items], 1, 5)
ours_rmse  = np.sqrt(np.mean((ours_preds - test_ratings) ** 2))
spW = np.mean(slim.W == 0)

print(f"Наш SLIM RMSE:           {ours_rmse:.4f}")
print(f"Разреженность матрицы W: {spW:.2%}")

Наш SLIM RMSE:           2.3393
Разреженность матрицы W: 99.50%


## 4. Эталон: KarypisLab/SLIM



In [6]:
R_sparse = csr_matrix(R_train)
trainmat  = SLIMatrix(R_sparse)

params   = {"algo": "cd", "nthreads": 4, "l1r": 1.0, "l2r": 1.0}
slim_ref = SLIMRef()

t0 = time.time()
slim_ref.train(params, trainmat)
ref_time = time.time() - t0

print(f"Время обучения: {ref_time:.1f} сек")

Learning takes 1.040 secs.
Время обучения: 1.0 сек


In [7]:
W_ref      = slim_ref.to_csr()
R_pred_ref = (R_sparse @ W_ref).toarray()

ref_preds = np.clip(R_pred_ref[test_users, test_items], 1, 5)
ref_rmse  = np.sqrt(np.mean((ref_preds - test_ratings) ** 2))

print(f"KarypisLab SLIM RMSE: {ref_rmse:.4f}")

KarypisLab SLIM RMSE: 2.4530


## 5. NDCG@10


In [8]:
def dcg_at_k(relevance, k):
    rel = np.array(relevance[:k], dtype=float)
    if len(rel) == 0:
        return 0.0
    return np.sum(rel / np.log2(np.arange(2, len(rel) + 2)))

def ndcg_at_k(actual, predicted, k=10):
    pred_order  = np.argsort(-predicted)
    ideal_order = np.argsort(-actual)
    dcg  = dcg_at_k(actual[pred_order],  k)
    idcg = dcg_at_k(actual[ideal_order], k)
    return dcg / idcg if idcg > 0 else 0.0

def mean_ndcg(R_true, R_pred, k=10):
    scores = []
    for u in range(R_true.shape[0]):
        rated = R_true[u] > 0
        if rated.sum() < 2:
            continue
        actual    = R_true[u, rated]
        predicted = R_pred[u, rated]
        scores.append(ndcg_at_k(actual, predicted, k=min(k, len(actual))))
    return float(np.mean(scores))

In [9]:
ours_ndcg = mean_ndcg(R_test, R_pred_ours, k=10)
ref_ndcg  = mean_ndcg(R_test, R_pred_ref,  k=10)

print(f"Наш SLIM NDCG@10:        {ours_ndcg:.4f}")
print(f"KarypisLab SLIM NDCG@10: {ref_ndcg:.4f}")

Наш SLIM NDCG@10:        0.9009
KarypisLab SLIM NDCG@10: 0.9017


## 6. Сводные результаты

In [10]:
results = pd.DataFrame({
    "Модель":           ["SLIM (своя реализация)", "SLIM (KarypisLab, эталон)"],
    "RMSE":             [f"{ours_rmse:.4f}",        f"{ref_rmse:.4f}"],
    "NDCG@10":          [f"{ours_ndcg:.4f}",        f"{ref_ndcg:.4f}"],
    "Время обучения":   [f"{our_time:.1f} сек",     f"{ref_time:.1f} сек"],
})
print(results.to_string(index=False))

                   Модель   RMSE NDCG@10 Время обучения
   SLIM (своя реализация) 2.3393  0.9009        0.9 сек
SLIM (KarypisLab, эталон) 2.4530  0.9017        1.0 сек
